# 04 · Pipeline de Mensageria — Backtest v3 + Estado Atual

**Objetivo:** combinar saídas dos notebooks 01 e 02, rodar o `trigger_engine.py` v3.0
(nova hierarquia RISCO / CRÍTICO / EMERGENCIAL) sobre o histórico completo (backtest) e avaliar estado atual.

| Etapa | Descrição |
|---|---|
| A | Carregar e preparar dados (01 + 02), montar df_hourly com DateTimeIndex |
| B | Backtest histórico: ciclo a ciclo, dia a dia |
| C | Linha do tempo — gatilhos × trocas reais |
| D | Estado atual — avaliação com os dados mais recentes |
| E | (Opcional) Escrever no SharePoint |
| F | Envio retroativo: backtest 2026 → SharePoint |

In [1]:
import sys, json, warnings, tempfile
from pathlib import Path
from datetime import datetime
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.trigger_engine import TriggerEngine, TriggerEvent

# ── Configurações ────────────────────────────────────────────────────────────
ARQUIVO_TROCAS    = 'troca_modulo.csv'
COLUNA_DATA_TROCA = 'Data-base do inicio'
MAQUINA           = 'FB14'
LIST_NAME         = 'Gatilhos_Selagem'
ESCREVER_SP       = False   # True → escreve no SharePoint (requer sharepoint.ev)
# ─────────────────────────────────────────────────────────────────────────────
print('Módulos OK')

Módulos OK


## Etapa A — Carregar e preparar df_hourly

In [2]:
v1 = pd.read_csv('01_vida_rul.csv',     parse_dates=['Timestamp'])
v2 = pd.read_csv('02_sinais_forca.csv', parse_dates=['Timestamp'])

v1['Timestamp'] = pd.to_datetime(v1['Timestamp'], utc=True)
v2['Timestamp'] = pd.to_datetime(v2['Timestamp'], utc=True)

# Merge: pega Media de v2 (recalculada) + colunas v2 para referência visual
df = v1.merge(
    v2[['Timestamp', 'Media', 'Delta_AB',
        'slope_Media_7d', 'mean_3d', 'mean_14d',
        'signal_score', 'proj_48h', 'p_risk']],
    on='Timestamp', how='left'
)

# DateTimeIndex — necessário para o trigger_engine
df = df.set_index('Timestamp').sort_index()

# Trocas
tc = pd.read_csv(ARQUIVO_TROCAS, parse_dates=[COLUNA_DATA_TROCA])
tc = tc.sort_values(COLUNA_DATA_TROCA).reset_index(drop=True)
troca_dates = pd.to_datetime(tc[COLUNA_DATA_TROCA]).dt.tz_localize('UTC').tolist()

print(f'df_hourly: {len(df)} linhas  |  {df.index.min().date()} -> {df.index.max().date()}')
print(f'Trocas   : {len(troca_dates)}  (ultima: {troca_dates[-1].date()})')
print(f'Colunas  : {list(df.columns)}')


df_hourly: 3243 linhas  |  2022-06-27 -> 2026-05-20
Trocas   : 14  (ultima: 2026-05-06)
Colunas  : ['horas_desde_troca', 'ciclo_id', 'score_weibull', 'score_roll7d', 'rul_p10', 'rul_p50', 'rul_p90', 'Media', 'Delta_AB', 'slope_Media_7d', 'mean_3d', 'mean_14d', 'signal_score', 'proj_48h', 'p_risk']


## Etapa B — Backtest histórico

Para cada ciclo histórico, simula o trigger engine dia a dia.  
Um arquivo de estado temporário é usado por ciclo para evitar interferência.

In [ ]:
all_events = []

state_path = Path(tempfile.mktemp(suffix='.json'))

for i, (t_ini, t_fim) in enumerate(zip(troca_dates[:-1], troca_dates[1:])):
    ciclo_df = df[(df.index >= t_ini) & (df.index < t_fim)]
    if ciclo_df.empty:
        continue

    if state_path.exists():
        state_path.unlink()
    engine = TriggerEngine(MAQUINA, state_path)

    dias = pd.DatetimeIndex(
        sorted({ts.normalize() for ts in ciclo_df.index})
    )

    for day in dias:
        df_ate_hoje = df[df.index <= day + pd.Timedelta(hours=23, minutes=59)]
        if df_ate_hoje.empty:
            continue
        try:
            events = engine.evaluate(
                df_ate_hoje,
                troca_date=t_ini.to_pydatetime(),
                sp_client=None,
                today=day,
                troca_dates=troca_dates,
            )
        except Exception as exc:
            print(f'  [ERRO] ciclo {i} dia {day.date()}: {exc}')
            continue

        for ev in events:
            all_events.append({
                'ciclo_id'           : i,
                'troca_ini'          : t_ini.date(),
                'troca_fim'          : t_fim.date(),
                'data_disparo'       : day.date(),
                'gatilho'            : ev.gatilho,
                'severidade'         : ev.severidade,
                'idade_dias'         : ev.idade_maintacker,
                'evento_no_ciclo'    : ev.evento_no_ciclo,
                'p_risk'             : ev.score_atual,
                'signal_score'       : ev.signal_score,
                'age_risk'           : ev.age_risk,
                'slope_7d'           : ev.slope_forca_7d,
                'mean_3d'            : ev.forca_minima_3d,
                'proj_48h'           : ev.proj_48h,
                'antecedencia_d'     : (t_fim.date() - day.date()).days,
            })

if state_path.exists():
    state_path.unlink()

eventos_df = pd.DataFrame(all_events).sort_values(by="troca_ini", ascending=False)
print(f'Total de eventos disparados no backtest : {len(eventos_df)}')
if not eventos_df.empty:
    print()
    print('Por tipo de gatilho:')
    print(eventos_df.groupby(['gatilho','severidade']).size().to_string())
    n_ciclos_fire = eventos_df['ciclo_id'].nunique()
    n_ciclos_tot  = len(troca_dates) - 1
    print()
    print(f'Ciclos cobertos : {n_ciclos_fire} / {n_ciclos_tot}  ({n_ciclos_fire/n_ciclos_tot:.0%})')

### Etapa B.1 — Preencher proj_48h via regressão age-gated

O backtest grava `proj_48h` apenas para gatilhos RED (EMERGENCIA/AMARELO/REVISAO ficam
com NaN). Além disso, o cálculo simples `mean_3d + slope*2` não tem gate de idade e é
ruído em maintackers jovens.

`proj_forca.adicionar_proj_48h_backtest()` recalcula via regressão OLS sobre a janela
configurável, **apenas para maintackers com ≥ 20 dias** (parâmetro `min_idade_dias` no
`config.yaml`). Valores existentes são preservados quando a regressão não produz resultado.

In [4]:
from src.proj_forca import adicionar_proj_48h_backtest

proj_series = adicionar_proj_48h_backtest(eventos_df, df)
eventos_df['proj_48h'] = proj_series

# Resumo do preenchimento
total   = len(eventos_df)
com_val = eventos_df['proj_48h'].notna().sum()
sem_val = total - com_val
por_gatilho = (
    eventos_df.groupby('gatilho')['proj_48h']
    .apply(lambda s: f"{s.notna().sum()}/{len(s)}")
)

print(f'proj_48h preenchida : {com_val}/{total} eventos  ({com_val/total:.0%})')
print(f'Sem valor (NaN)     : {sem_val}  (maintackers < 20 dias ou dados insuficientes)')
print()
print('Cobertura por tipo de gatilho:')
for gatilho, cobertura in por_gatilho.items():
    print(f'  {gatilho:<12} {cobertura}')
print()
print('Distribuição dos valores calculados (N):')
print(eventos_df['proj_48h'].describe().round(1).to_string())


proj_48h preenchida : 131/174 eventos  (75%)
Sem valor (NaN)     : 43  (maintackers < 20 dias ou dados insuficientes)

Cobertura por tipo de gatilho:
  AMARELO      51/61
  EMERGENCIA   41/56
  OUTLIER_SINAL 13/31
  RED          7/7
  REVISAO      19/19

Distribuição dos valores calculados (N):
count     131.0
mean      978.1
std       232.5
min       654.3
25%       814.2
50%       950.7
75%      1092.4
max      1782.5


In [5]:
eventos_df[eventos_df['severidade']=='CRITICA']

,ciclo_id,troca_ini,troca_fim,data_disparo,gatilho,severidade,idade_dias,p_risk,signal_score,age_risk,slope_7d,mean_3d,proj_48h,antecedencia_d
157,10,2026-02-02,2026-03-23,2026-02-18,EMERGENCIA,CRITICA,16,0.4601,NaN,NaN,-50.9,721.5,NaN,33
145,9,2026-01-06,2026-02-02,2026-01-13,EMERGENCIA,CRITICA,7,0.0977,NaN,NaN,-2.6,756.4,NaN,20
146,9,2026-01-06,2026-02-02,2026-01-15,EMERGENCIA,CRITICA,9,0.1134,NaN,NaN,16.7,756.4,NaN,18
144,8,2025-12-18,2026-01-06,2026-01-05,EMERGENCIA,CRITICA,18,0.3641,NaN,NaN,-22.9,570.0,NaN,1
142,8,2025-12-18,2026-01-06,2026-01-03,EMERGENCIA,CRITICA,16,0.4114,NaN,NaN,-39.0,710.7,NaN,3
141,8,2025-12-18,2026-01-06,2025-12-29,EMERGENCIA,CRITICA,11,0.1415,NaN,NaN,12.6,763.2,NaN,8
140,8,2025-12-18,2026-01-06,2025-12-27,EMERGENCIA,CRITICA,9,0.1240,NaN,NaN,10.9,763.2,NaN,10
139,8,2025-12-18,2026-01-06,2025-12-23,EMERGENCIA,CRITICA,5,0.1541,NaN,NaN,-13.2,682.0,NaN,14
132,7,2025-11-24,2025-12-18,2025-12-07,EMERGENCIA,CRITICA,13,0.1988,NaN,NaN,-3.8,686.7,NaN,11
131,7,2025-11-24,2025-12-18,2025-12-05,EMERGENCIA,CRITICA,11,0.3345,NaN,NaN,-36.6,734.0,NaN,13


In [6]:
if not eventos_df.empty:
    primeiro = (
        eventos_df
        .sort_values('data_disparo')
        .groupby('ciclo_id', as_index=False)
        .first()
    )
    print('Primeiro disparo RED por ciclo:')
    display(
        primeiro[['ciclo_id', 'troca_ini', 'troca_fim', 'data_disparo',
                  'antecedencia_d', 'idade_dias', 'p_risk', 'signal_score', 'proj_48h', 'gatilho']]
        .sort_values('ciclo_id')
        .reset_index(drop=True)
    )


Primeiro disparo RED por ciclo:


,ciclo_id,troca_ini,troca_fim,data_disparo,antecedencia_d,idade_dias,p_risk,signal_score,proj_48h,gatilho
0,1,2025-02-28,2025-04-22,2025-03-13,40,13,0.1994,0.0000,1235.166752,OUTLIER_SINAL
1,2,2025-04-22,2025-05-21,2025-05-04,17,12,0.3297,0.5116,786.073885,OUTLIER_SINAL
2,3,2025-05-21,2025-07-24,2025-05-27,58,6,0.1225,0.0000,994.270113,OUTLIER_SINAL
3,6,2025-08-13,2025-11-24,2025-08-18,98,5,0.0639,0.4322,727.727605,OUTLIER_SINAL
4,7,2025-11-24,2025-12-18,2025-12-03,15,9,0.2454,0.3026,977.991415,OUTLIER_SINAL
5,8,2025-12-18,2026-01-06,2025-12-23,14,5,0.1541,0.3901,NaN,EMERGENCIA
6,9,2026-01-06,2026-02-02,2026-01-13,20,7,0.0977,0.1697,1100.764397,EMERGENCIA
7,10,2026-02-02,2026-03-23,2026-02-18,33,16,0.4601,0.4852,1002.742139,EMERGENCIA
8,11,2026-03-23,2026-04-13,2026-04-09,4,17,0.2836,0.3367,1060.841577,OUTLIER_SINAL
9,12,2026-04-13,2026-05-06,2026-04-28,8,15,0.3494,0.2911,965.104280,AMARELO


## Etapa C — Linha do tempo: gatilhos × trocas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def force_utc(t):
    ts = pd.to_datetime(t)
    return ts.tz_localize('UTC') if ts.tz is None else ts.tz_convert('UTC')

inicio = force_utc('2026-04-01')
fim    = force_utc(f'{force_utc("2026-12-31 23:59:59").year}-12-31 23:59:59')

if df.index.tz is None:
    df.index = df.index.tz_localize('UTC')
else:
    df.index = df.index.tz_convert('UTC')

df_filt = df.loc[inicio:fim].copy()

eventos_df['data_disparo'] = eventos_df['data_disparo'].apply(force_utc)
eventos_filt = eventos_df[
    (eventos_df['data_disparo'] >= inicio) & (eventos_df['data_disparo'] <= fim)
].copy()
troca_dates_filt = [t for t in troca_dates if inicio <= force_utc(t) <= fim]

# Severidade → cor (v3.0: ALTA adicionada para CRITICO)
mapa_cores = {
    'INFO':   "#e9f00c",   # amarelo — RISCO / REVISAO
    'MEDIA':  "#2e0cd9",   # azul    — AMARELO
    'ALTA':   "#ff8c00",   # laranja — CRITICO / RED
    'CRITICA':"#ff0000",   # vermelho — EMERGENCIAL
}
cor_troca = '#39ff14'

plt.style.use('dark_background')
fig, axes = plt.subplots(2, 1, figsize=(22, 10), sharex=True, facecolor='black')

for i, painel in enumerate(['Media', 'p_risk']):
    ax = axes[i]
    ax.set_facecolor('black')
    ax.plot(df_filt.index, df_filt[painel], lw=0.8, color='white', alpha=0.3)

    for j, t in enumerate(troca_dates_filt):
        ax.axvline(force_utc(t), color=cor_troca, lw=6, alpha=0.4,
                   label='Troca Real' if (i == 0 and j == 0) else "")

    if not eventos_filt.empty:
        for sev, grupo in eventos_filt.groupby('severidade'):
            cor = mapa_cores.get(sev, 'white')
            indices = df_filt.index.get_indexer(grupo['data_disparo'], method='nearest')
            x_pts = df_filt.index[indices]
            y_pts = df_filt.iloc[indices][painel]
            ax.scatter(x_pts, y_pts, color=cor, s=120, zorder=10,
                       edgecolors='white', lw=0.5,
                       label=f'{sev}' if i == 0 else "")

axes[0].set_title('FB14 — Gatilhos v3.0 × Trocas (RISCO / CRÍTICO / EMERGENCIAL)', fontsize=14)
axes[0].set_ylabel('Força Média (N)')
axes[1].set_ylabel('Prob. Risco (p_risk)')
axes[1].set_ylim(-0.05, 1.1)
axes[0].legend(loc='upper right', bbox_to_anchor=(1, 1.15), ncol=6,
               frameon=True, facecolor='#222')

plt.tight_layout()
plt.show()

## Etapa D — Estado atual

In [ ]:
from src.trigger_engine import compute_p_risk_snapshot

ultima_troca = troca_dates[-1]
hoje = df.index.max().normalize()

state_atual = Path('state_fb14.json')
engine_atual = TriggerEngine(MAQUINA, state_atual)

df_atual = df[df.index >= ultima_troca]

print(f'Ultima troca : {ultima_troca.date()}')
print(f'Avaliando em : {hoje.date()}')
print(f'Dados no ciclo atual : {len(df_atual)} linhas')
print()

if df_atual.empty:
    print('Ciclo atual sem dados — não é possível avaliar.')
else:
    snap = compute_p_risk_snapshot(df, ultima_troca.to_pydatetime(), hoje)
    print('── Snapshot p_risk (engine v3) ─────────────────────────')
    for k, v in snap.items():
        flag = ''
        if k == 'cond_p_risk'  and v: flag = '  ✓ COND1'
        if k == 'cond_signal'  and v: flag = '  ✓ COND2'
        if k == 'cond_idade'   and v: flag = '  ✓ COND3'
        if k == 'cond_proj'    and v: flag = '  ✓ COND4'
        if k == 'cond_critico' and v: flag = '  ✓ FORÇA CRÍTICA'
        if k == 'cond_risco'   and v: flag = '  ✓ OUTLIER ISOLADO'
        print(f'  {k:<20}: {v}{flag}')
    print()

    eventos_hoje = engine_atual.evaluate(
        df,
        troca_date=ultima_troca.to_pydatetime(),
        sp_client=None,
        today=hoje,
        troca_dates=troca_dates,
    )

    if eventos_hoje:
        print(f'  GATILHO(S) DISPARADO(S) ({len(eventos_hoje)}):')
        for ev in eventos_hoje:
            print(f'    [{ev.severidade}] {ev.gatilho}  evento_no_ciclo={ev.evento_no_ciclo}')
            print(f'    {ev.mensagem}')
    else:
        print('  Nenhum gatilho disparado hoje — situação NORMAL.')

    print()
    print('Estado do engine (state_fb14.json):')
    with open(state_atual) as f:
        print(json.dumps(json.load(f), indent=2))

## Etapa E — Escrever no SharePoint (opcional)

Executar somente quando `ESCREVER_SP = True` na célula de configuração.  
Requer `sharepoint.ev` com `SP_USER=` e `SP_PASS=`.

if not ESCREVER_SP:
    print('ESCREVER_SP=False -- pulando escrita no SharePoint.')
    print('Para ativar: set ESCREVER_SP = True na celula de configuracao.')
else:
    from src.sharepoint_methods import SharePointClient
    from dotenv import dotenv_values

    creds = dotenv_values('../sharepoint.ev')
    sp_user = creds.get('SP_USER')
    sp_pass = creds.get('SP_PASS')
    sp_url  = creds.get('SP_URL', 'https://kimberlyclark.sharepoint.com/Sites/H945')

    if not sp_user or not sp_pass:
        raise RuntimeError('Credenciais SP_USER/SP_PASS nao encontradas em sharepoint.ev')

    sp_client = SharePointClient(url=sp_url, username=sp_user, password=sp_pass)

    engine_sp = TriggerEngine(MAQUINA, Path('state_fb14.json'))
    eventos_sp = engine_sp.evaluate(
        df,
        troca_date=ultima_troca.to_pydatetime(),
        sp_client=sp_client,
        list_name=LIST_NAME,
        today=hoje,
    )

    if eventos_sp:
        print(f'{len(eventos_sp)} evento(s) escrito(s) na lista "{LIST_NAME}":')
        for ev in eventos_sp:
            print(f'  [{ev.severidade}] {ev.gatilho}  SP_ID={ev.sp_item_id}')
    else:
        print('Nenhum evento para escrever -- estado atual normal.')

## Resumo do backtest

In [ ]:
if not eventos_df.empty:
    print('=' * 60)
    print('  RESUMO DO BACKTEST v3 — Todos os Gatilhos')
    print('=' * 60)
    n_ciclos = len(troca_dates) - 1
    cobertos = eventos_df['ciclo_id'].nunique()
    print(f'  Ciclos historicos avaliados : {n_ciclos}')
    print(f'  Total de eventos disparados : {len(eventos_df)}')
    print(f'  Ciclos com disparo          : {cobertos} / {n_ciclos}  ({cobertos/n_ciclos:.0%})')
    print()
    print('  Por tipo:')
    for (gat, sev), n in eventos_df.groupby(['gatilho','severidade']).size().items():
        print(f'    {gat:<14} [{sev}]  {n}')
    print()

    red_df = eventos_df[eventos_df['gatilho'] == 'RED']
    if not red_df.empty:
        primeiro_red = (
            red_df
            .sort_values('data_disparo')
            .groupby('ciclo_id', as_index=False)
            .first()
        )
        vals_ant = primeiro_red['antecedencia_d']
        vals_p   = primeiro_red['p_risk']
        print(f'  Antecedencia do 1o disparo RED:')
        print(f'    median = {vals_ant.median():.0f} dias')
        print(f'    min    = {vals_ant.min():.0f} dias')
        print(f'    max    = {vals_ant.max():.0f} dias')
        print()
        print(f'  p_risk no 1o disparo RED por ciclo:')
        print(f'    median = {vals_p.median():.3f}')
        print(f'    min    = {vals_p.min():.3f}')
        print(f'    max    = {vals_p.max():.3f}')
        print()

    todos_ciclos = set(range(len(troca_dates) - 1))
    ciclos_fire  = set(eventos_df['ciclo_id'].unique())
    ciclos_sil   = sorted(todos_ciclos - ciclos_fire)
    if ciclos_sil:
        print(f'  Ciclos silenciosos (sem disparo): {ciclos_sil}')
        print('  (ciclos curtos = trocas preventivas — comportamento esperado)')
    print('=' * 60)
else:
    print('Nenhum evento disparado no backtest.')

## Etapa F — Envio retroativo: backtest 2026 → SharePoint

Envia todos os eventos de `eventos_filt` (2026) para a lista `Gatilhos_Selagem`,
gerando o `TeamsPayload` (Adaptive Card) para cada alerta.

Pré-requisito: `sharepoint.ev` com `SP_USER=` e `SP_PASS=` na raiz do projeto.

In [10]:
eventos_filt['data_disparo'] = pd.to_datetime(eventos_filt['data_disparo'])

eventos_filt = eventos_filt[
    (eventos_filt['data_disparo'].dt.year == 2026) &
    (eventos_filt['data_disparo'].dt.month >= 4)
]

eventos_filt = eventos_filt.sort_values(by='data_disparo', ascending=True)
eventos_filt

,ciclo_id,troca_ini,troca_fim,data_disparo,gatilho,severidade,idade_dias,p_risk,signal_score,age_risk,slope_7d,mean_3d,proj_48h,antecedencia_d
167,11,2026-03-23,2026-04-13,2026-04-09 00:00:00+00:00,OUTLIER_SINAL,INFO,17,0.2836,NaN,NaN,-11.9,797.7,NaN,4
169,11,2026-03-23,2026-04-13,2026-04-11 00:00:00+00:00,AMARELO,MEDIA,19,0.4161,0.3367,NaN,-34.9,1102.7,NaN,2
168,11,2026-03-23,2026-04-13,2026-04-11 00:00:00+00:00,OUTLIER_SINAL,INFO,19,0.4161,NaN,NaN,-34.9,797.7,NaN,2
170,11,2026-03-23,2026-04-13,2026-04-12 00:00:00+00:00,REVISAO,INFO,20,0.4634,0.4139,NaN,-47.5,1123.9,1060.841577,1
171,12,2026-04-13,2026-05-06,2026-04-28 00:00:00+00:00,AMARELO,MEDIA,15,0.3494,0.2911,NaN,-24.1,1019.1,NaN,8
172,12,2026-04-13,2026-05-06,2026-05-01 00:00:00+00:00,AMARELO,MEDIA,18,0.3260,0.1760,NaN,-14.6,1006.6,NaN,5
173,12,2026-04-13,2026-05-06,2026-05-03 00:00:00+00:00,REVISAO,INFO,20,0.2835,0.0368,NaN,19.9,1013.8,965.104280,3


In [ ]:
from dotenv import dotenv_values
from src.sharepoint_methods import SharePointClient
from src.card_formatter import build_alert_card
import json, numpy as np

creds     = dotenv_values('../sharepoint.ev')
LIST_NAME = 'Gatilhos_Selagem'
BATCH_SIZE = 10

_ACAO = {
    'RISCO':       ('Ir verificar no local — analisar última amostra e identificar causa '
                    'da força extremamente baixa. '
                    'Se 2+ leituras abaixo de 800 N nas próximas 72h, o sistema escalará para CRÍTICO.'),
    'CRITICO':     ('Análise aprofundada e planejamento de troca do rolo maintacker.'),
    'EMERGENCIAL': ('Risco iminente de retenção — acionar troca imediata do rolo maintacker.'),
    'RED':         ('Programar inspeção preventiva do rolo maintacker. '
                    'Verificar força de selagem no próximo turno. '
                    'Registrar: OK / Troca programada / Troca imediata.'),
    'AMARELO':     ('Monitorar força de selagem diariamente. '
                    'Incluir troca do maintacker no próximo plano de manutenção preventiva.'),
    'REVISAO':     ('Revisar tendência de força no gráfico. '
                    'Se forças abaixo de 900 N ou tendência negativa persistente, '
                    'antecipar inspeção do rolo.'),
}

_NOVOS_NIVEIS = {'RISCO', 'CRITICO', 'EMERGENCIAL'}

def _nan_to_none(x):
    if x is None: return None
    try: return None if np.isnan(float(x)) else float(x)
    except (TypeError, ValueError): return None

def _build_retroativo_json(row: dict, acao: str) -> str:
    """JSON estruturado v3.0 para envio retroativo (campos disponíveis no backtest)."""
    antecedencia = int(row.get('antecedencia_d', 0))
    d = {
        'nivel':             row['gatilho'],
        'evento_no_ciclo':   int(row.get('evento_no_ciclo', 0)),
        'dias_operacao':     int(row['idade_dias']),
        'eta_ajustado_dias': int(row['idade_dias']) + antecedencia,
        'dias_restantes':    antecedencia,
        'acao':              acao,
    }
    forca = _nan_to_none(row.get('mean_3d'))
    proj  = _nan_to_none(row.get('proj_48h'))
    if row['gatilho'] == 'RISCO':
        d['data_evento'] = str(row['data_disparo'])
        if forca: d['forca_gf'] = int(round(forca))
    elif row['gatilho'] == 'CRITICO':
        d['data_evento'] = str(row['data_disparo'])
        if forca: d['forca_min_ultima_amostra_gf'] = int(round(forca))
        if proj:  d['forca_projetada_gf'] = int(round(proj))
        d['p_risk'] = round(float(row.get('p_risk', 0) or 0), 2)
    elif row['gatilho'] == 'EMERGENCIAL':
        if forca: d['forca_min_ultima_amostra_gf'] = int(round(forca))
        if proj:  d['forca_projetada_48h_gf'] = int(round(proj))
        d['p_risk'] = round(float(row.get('p_risk', 0) or 0), 2)
    return json.dumps(d, ensure_ascii=False)

sp = SharePointClient("https://kimberlyclark.sharepoint.com/Sites/H945",
                      creds['SP_USER'], creds['SP_PASS'])
print('SharePoint conectado.')

df_existentes   = sp.query_large_list(LIST_NAME)
titles_exist    = set(df_existentes['Title'].dropna()) if 'Title' in df_existentes.columns else set()
print(f'Itens já na lista      : {len(titles_exist)}')
print(f'Eventos 2026 no backtest: {len(eventos_filt)}')

itens   = []
pulados = []
for _, ev in eventos_filt.iterrows():
    gatilho = ev['gatilho']
    data_ts = ev['data_disparo']
    title   = f"{MAQUINA} | {gatilho} | {pd.Timestamp(data_ts).strftime('%Y-%m-%d')}"

    if title in titles_exist:
        pulados.append(title)
        continue

    acao = _ACAO.get(gatilho, 'Avaliar situação com equipe de manutenção.')

    if gatilho in _NOVOS_NIVEIS:
        payload = _build_retroativo_json(ev.to_dict(), acao)
    else:
        payload = build_alert_card(
            maquina          = MAQUINA,
            gatilho          = gatilho,
            idade_dias       = int(ev['idade_dias']),
            p_risk           = float(ev['p_risk']),
            slope_7d         = _nan_to_none(ev['slope_7d']),
            forca_min_3d     = _nan_to_none(ev['mean_3d']),
            proj_48h         = _nan_to_none(ev['proj_48h']),
            acao_recomendada = acao,
            data_disparo     = pd.Timestamp(data_ts).to_pydatetime(),
        )
    itens.append({'Title': title, 'Maquina': MAQUINA, 'TeamsPayload': payload})

print(f'Já existentes (pulados): {len(pulados)}')
print(f'Novos a inserir        : {len(itens)}')

if not itens:
    print('Nenhum item novo — lista já está atualizada.')
else:
    ids_criados = []
    for i in range(0, len(itens), BATCH_SIZE):
        lote = itens[i : i + BATCH_SIZE]
        ids  = sp.insert_list_item(LIST_NAME, lote)
        ids_criados.extend(ids)
        datas = [item['Title'].split(' | ')[-1] for item in lote]
        print(f'  Lote {i//BATCH_SIZE+1:02d}: {len(lote)} itens '
              f'({datas[0]} → {datas[-1]})  IDs: {ids}')
    print(f'\n✓ Total inserido: {len([x for x in ids_criados if x])} / {len(itens)}')